## Orchestrateur

Ce notebook a deux parties :
- **Partie A** : tous les agents sont factices. Elle vérifie le routage, les règles de décision et la trace.
- **Partie B** : `classify` est réel et appelle le LLM. Les autres agents restent factices.

Après avoir modifié un fichier de `src/`, redémarre le noyau avant de relancer.

In [ ]:
!pip install langgraph langchain-openai

In [ ]:
import json
import os
import sys

import pandas as pd

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)

from src.graph import build_graph
from src.trace import show_trace, trace_rows

with open(os.path.join(ROOT, "tests", "scenarios.json"), encoding="utf-8") as f:
    SCENARIOS = json.load(f)["scenarios"]

print(len(SCENARIOS), "scénarios chargés")

In [ ]:
def run(scenario, g):
    return g.invoke({
        "ticket": scenario["ticket"],
        "fixtures": scenario["fixtures"],
        "trace": [],
    })


def show(scenario_id, g):
    scenario = next(s for s in SCENARIOS if s["id"] == scenario_id)
    result = run(scenario, g)
    print(scenario["enonce"])
    print("TICKET :", scenario["ticket"]["description"])
    show_trace(result["trace"])
    print("\nISSUE :", result["decision"], "| règle :", result["decision_rule"])
    print(result["answer"])
    return result


def check_all(g):
    rows = []
    for s in SCENARIOS:
        r = run(s, g)
        exp = s["expected"]
        rows.append({
            "scénario": s["id"],
            "attendu": exp["decision"],
            "obtenu": r["decision"],
            "règle attendue": exp["rule"],
            "règle obtenue": r["decision_rule"],
            "thème": r.get("theme"),
            "cas de prix": r.get("price_case"),
            "parcours": " > ".join(e["noeud"] for e in r["trace"]),
            "OK": r["decision"] == exp["decision"] and r["decision_rule"] == exp["rule"],
        })
    bilan = pd.DataFrame(rows)
    print(f"{int(bilan['OK'].sum())} / {len(bilan)} scénarios conformes")
    return bilan

## Partie A : agents factices

In [ ]:
graph = build_graph()
result = show("S2_prix_plusieurs_donnees", graph)

In [ ]:
pd.DataFrame(trace_rows(result["ticket"]["ticket_id"], result["trace"]))

In [ ]:
check_all(graph)

In [ ]:
import importlib
import sys

from IPython.display import Image, display

# Recharge les modules de src/ pour ne pas dessiner une ancienne version
# gardée en mémoire par le noyau.
for name in sorted((m for m in list(sys.modules) if m == "src" or m.startswith("src.")),
                   key=len, reverse=True):
    importlib.reload(sys.modules[name])

import src.graph

graph_dessin = src.graph.build_graph()
noeuds = [n for n in graph_dessin.get_graph().nodes if not n.startswith("__")]
print("Nœuds du graphe :", " > ".join(noeuds))

if "resolve_refs" not in noeuds:
    print("\nATTENTION : le src/graph.py du workspace est une ancienne version "
          "(pas de nœud resolve_refs). Remplace-le par la version à jour.")

try:
    display(Image(graph_dessin.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Image indisponible (", type(e).__name__, ") : schéma en texte Mermaid ci-dessous\n")
    print(graph_dessin.get_graph().draw_mermaid())

## Partie B : `classify` réel

Le LLM lit le ticket et trouve le thème, le périmètre, l'équipe et le besoin de données. Les exemples qu'on lui montre viennent de `KB_QA`.

In [ ]:
from snowflake.snowpark.context import get_active_session

from src.llm import MODEL, make_llm
from src.nodes.classify import build_prompt, load_examples, make_classify

session = get_active_session()
llm = make_llm()

examples = load_examples(session)
print(len(examples), "exemples tirés de KB_QA")

graph_reel = build_graph(llm=llm, session=session)

### Le prompt envoyé au LLM

In [ ]:
print(build_prompt(examples))

### Un scénario avec le vrai `classify`

Regarde l'étape 2 de la trace : l'outil n'est plus marqué « factice ».

In [ ]:
result = show("S2_prix_plusieurs_donnees", graph_reel)

### Les sept scénarios avec le vrai `classify`

L'issue dépend maintenant de ce que le LLM comprend du ticket. Un scénario non conforme n'est pas forcément un bug : il montre ce qu'il faut ajuster dans le prompt.

In [ ]:
check_all(graph_reel)

### Mesure sur 40 questions de `KB_QA`

On compare le thème et le périmètre trouvés par le LLM aux étiquettes du fichier. Les questions déjà montrées en exemple sont exclues. Cette mesure sert à régler le prompt ; l'évaluation finale se fera sur `EVAL_TICKETS`.

In [ ]:
classify_node = make_classify(llm, examples, MODEL)

kb = session.table("PROJECT_DB.PUBLIC.KB_QA").to_pandas()
example_ids = {e["id"] for e in examples}
dev = kb[~kb["ID"].isin(example_ids)].sample(40, random_state=0)

rows = []
for _, r in dev.iterrows():
    out = classify_node({"ticket": {"ticket_id": r["ID"], "description": r["QUESTION"]}})
    rows.append({
        "ID": r["ID"],
        "thème attendu": r["THEME"],
        "thème obtenu": out["theme"],
        "périmètre attendu": r["STATUS"] != "Out of Scope",
        "périmètre obtenu": out["in_scope"],
        "besoins": ", ".join(out["needs"]),
        "cas de prix": out["price_case"],
        "question": r["QUESTION"][:90],
    })

mesure = pd.DataFrame(rows)
theme_ok = mesure["thème attendu"] == mesure["thème obtenu"]
scope_ok = mesure["périmètre attendu"] == mesure["périmètre obtenu"]
print(f"Thème correct     : {int(theme_ok.sum())} / {len(mesure)}")
print(f"Périmètre correct : {int(scope_ok.sum())} / {len(mesure)}")

mesure[~(theme_ok & scope_ok)]

In [ ]:
%%sql -r dataframe_1


In [ ]:
%%sql -r dataframe_2
SELECT
    COUNT(*) AS NB_PASSAGES,
    COUNT(EMBEDDINGS) AS AVEC_EMBEDDING,
    COUNT(DISTINCT SPLIT_PART(FILE_NAME, '#', 1)) AS NB_FICHIERS
FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG;

In [ ]:
%%sql -r dataframe_3
SELECT
    SPLIT_PART(FILE_NAME, '#', 1) AS FICHIER,
    COUNT(*) AS NB_PASSAGES
FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
GROUP BY 1
ORDER BY 1;

In [ ]:
%%sql -r dataframe_4
DELETE FROM PROJECT_DB.PUBLIC.BASE_CONNAISSANCE_RAG
WHERE FILE_NAME ILIKE '%.xlsx#%';